In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import s3fs
import torch
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from config import MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY, BUCKET_NAME

# Cấu hình để vẽ hình ngay trong Notebook
%matplotlib inline
warnings.filterwarnings('ignore')

# --- CẤU HÌNH HỆ THỐNG (Sửa đường dẫn nếu cần) ---
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.31.11-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

print("✅ Đã nạp cấu hình môi trường thành công!")

In [ ]:
print("📊 Đang kết nối MinIO và tải tập dữ liệu Gold...")

try:
    fs = s3fs.S3FileSystem(
        client_kwargs={'endpoint_url': MINIO_ENDPOINT}, 
        key=MINIO_ACCESS_KEY, 
        secret=MINIO_SECRET_KEY
    )
    
    path = f"{BUCKET_NAME}/gold/features.parquet"
    df = pd.read_parquet(path, filesystem=fs)
    
    # Đảm bảo tên cột mục tiêu khớp với các bước trước
    if "PM2_5" not in df.columns and "PM2.5" in df.columns:
        df = df.rename(columns={"PM2.5": "PM2_5"})
        
    print(f"✅ Tải dữ liệu thành công! Kích thước: {df.shape}")
    display(df.head()) # Hiển thị 5 dòng đầu dạng bảng đẹp
except Exception as e:
    print(f"❌ Lỗi kết nối hoặc đọc file: {e}")

Định nghĩa cấu trúc mô hình LSTM

In [ ]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=100, num_layers=3, target_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, 
                            num_layers=num_layers, batch_first=True, 
                            dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, target_size)
        
    def forward(self, x):
        ula, (h_out, _) = self.lstm(x)
        out = self.fc(ula[:, -1, :])
        return out

def create_sequences_test(input_data, features_columns):
    seq_length = 48
    numeric_df = input_data[features_columns].select_dtypes(include=[np.number])
    np_data = numeric_df.values
    sequences = []
    for i in range(len(np_data) - seq_length):
        sequences.append(np_data[i : i + seq_length])
    return np.array(sequences)

print("✅ Đã định nghĩa cấu trúc LSTM và hàm tạo chuỗi.")

Thực hiện Dự đoán và So sánh

In [ ]:
# 1. Chuẩn bị tập Test
target_col = "PM2_5"
features_columns = [col for col in df.columns if col not in ['No', 'station', 'wd', target_col]]
_, test_df = train_test_split(df, test_size=0.1, shuffle=False)

# Cắt 48 dòng đầu để đồng bộ với cửa sổ trượt của LSTM
test_tree = test_df.iloc[48:].copy()
y_true = test_tree[target_col].values

print(f"🔮 Đang bắt đầu dự đoán trên {len(y_true)} mẫu thử nghiệm...")

# --- DỰ ĐOÁN ---
# XGBoost
xgb_model = xgb.Booster()
xgb_model.load_model("model_xgboost_pm25.json")
y_pred_xgb = xgb_model.predict(xgb.DMatrix(test_tree[features_columns]))

# LightGBM
lgb_model = lgb.Booster(model_file="model_lightgbm_pm25.txt")
y_pred_lgb = lgb_model.predict(test_tree[features_columns])

# LSTM
device = "cuda" if torch.cuda.is_available() else "cpu"
x_test_lstm = create_sequences_test(test_df, features_columns)
num_features = x_test_lstm.shape[2]

lstm_model = LSTMModel(input_size=num_features).to(device)
lstm_model.load_state_dict(torch.load("model_lstm_pm25.pth", map_location=device))
lstm_model.eval()

with torch.no_grad():
    y_pred_lstm = lstm_model(torch.tensor(x_test_lstm).float().to(device)).squeeze().cpu().numpy()

# --- HIỂN THỊ CHỈ SỐ ---
results = []
models = {"XGBoost": y_pred_xgb, "LightGBM": y_pred_lgb, "LSTM": y_pred_lstm}
for name, y_pred in models.items():
    mse = mean_squared_error(y_true, y_pred)
    results.append({
        "Model": name, "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(y_true, y_pred)
    })

print("\n📊 BẢNG SO SÁNH CHỈ SỐ:")
display(pd.DataFrame(results))

# --- VẼ BIỂU ĐỒ ---
plt.figure(figsize=(15, 6))
plot_len = 150 
plt.plot(y_true[-plot_len:], label="Thực tế", color='black', linewidth=2)
plt.plot(y_pred_xgb[-plot_len:], label="XGBoost", linestyle='--')
plt.plot(y_pred_lgb[-plot_len:], label="LightGBM", linestyle='-.')
plt.plot(y_pred_lstm[-plot_len:], label="LSTM", color='red')
plt.title(f"So sánh dự báo PM2.5 trong {plot_len} giờ cuối", fontsize=15)
plt.legend()
plt.show()